# Laboratório 10 — O Pipeline Definitivo (RAG + QLoRA + Otimização de Inferência)

**Disciplina:** Tópicos em IA — iCEV
**Aluno:** Aderson Mesquita

> Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por **Aderson Mesquita**.

Pipeline ponta a ponta que demonstra como **QLoRA (4-bit)**, **KV Cache** e a **atencao eficiente em memoria** resolvem o colapso de VRAM causado pela complexidade O(n^2) do Self-Attention em contextos massivos recuperados por RAG.

> ⚙️ **Ambiente:** este notebook esta ajustado para a **GPU T4 gratuita do Google Colab**. Como a T4 (arquitetura Turing) NAO suporta FlashAttention-2, usamos o **PyTorch SDPA** (`attn_implementation="sdpa"`), que aplica o mesmo principio *hardware-aware* de nao materializar a matriz O(n^2) e e o substituto mais proximo disponivel. Justificativa completa no README.
>
> **Como rodar:** Ambiente de execucao -> Alterar tipo -> GPU **T4** -> Executar tudo.

## Setup — instalação das bibliotecas

In [ ]:
# Ambiente: GPU T4 (Colab gratuito). NAO instalamos flash-attn pois a T4 nao suporta FA2.
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os
# Reduz a fragmentacao de memoria da GPU (ajuda a evitar OOM na T4)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, time, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Ative a GPU: Ambiente de execucao -> Alterar tipo -> T4."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

def limpar_memoria():
    """Coleta lixo, esvazia o cache da GPU e zera o contador de pico de VRAM."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Passo 1 — Ingestão Eficiente (QLoRA 4-bit)

Carregamos o modelo gerador auto-regressivo quantizado em **4 bits** com `bitsandbytes`. Em FP16 ele ocuparia VRAM demais; em 4-bits cabe em poucas centenas de MB, liberando memoria para o contexto gigante. O `compute_dtype=float16` mantem a precisao no calculo.

> Carregamos com `attn_implementation="eager"` **de proposito** — esta e a versao *sem* otimizacao de hardware, que materializa a matriz de atencao O(n^2) na VRAM (o "desastre").

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configuracao QLoRA: pesos em 4-bits, calculo interno em float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",         # NormalFloat4 (padrao do paper QLoRA)
    bnb_4bit_use_double_quant=True,    # quantizacao dupla -> economia extra de VRAM
)

limpar_memoria()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    attn_implementation="eager",       # baseline NAO otimizado
)

vram_modelo = torch.cuda.memory_allocated() / 1024**2
print(f"[METRICA 1] VRAM do modelo quantizado em 4-bits: {vram_modelo:.2f} MB")

## Passo 2 — Simulando o RAG Massivo

Geramos um texto medico ficticio (permitido pelo enunciado) repetido ate ~6.000 tokens, simulando os capitulos recuperados pelo banco vetorial, e o tokenizamos.

> O TinyLlama tem janela nativa de 2048 tokens. Acima disso o texto gerado e ruido, mas o **benchmark de tempo e VRAM continua valido** — e a atencao que estamos medindo.

In [ ]:
# Paragrafo medico ficticio (gerado por IA)
paragrafo_medico = (
    "Paciente do sexo masculino, 58 anos, admitido na unidade de terapia intensiva "
    "com quadro de insuficiencia cardiaca congestiva descompensada e fracao de ejecao "
    "reduzida estimada em 32 por cento. Ao exame, apresentava estertores crepitantes em "
    "bases pulmonares, turgencia jugular patologica e edema periferico bilateral grau 3. "
    "A gasometria arterial evidenciou acidose metabolica compensada, e o ecocardiograma "
    "transtoracico confirmou disfuncao diastolica do ventriculo esquerdo. Iniciou-se "
    "terapia com diureticos de alca endovenosos, vasodilatadores e betabloqueadores "
    "titulados conforme protocolo institucional, com monitorizacao hemodinamica continua "
    "e controle rigoroso do balanco hidrico e dos niveis sericos de potassio e creatinina. "
)

TOKENS_ALVO = 6000    # reduzido p/ caber na VRAM da T4 (16GB); ver nota no README
tokens_por_paragrafo = len(tokenizer(paragrafo_medico).input_ids)
repeticoes = TOKENS_ALVO // tokens_por_paragrafo + 1

contexto_rag = " ".join([paragrafo_medico] * repeticoes)
inputs = tokenizer(contexto_rag, return_tensors="pt").to(model.device)

num_tokens = inputs.input_ids.shape[1]
print(f"Tokens por paragrafo: {tokens_por_paragrafo}")
print(f"[METRICA 2] Contexto RAG simulado: {num_tokens} tokens")

## Passo 3 — O Gargalo de Geração (sem KV Cache)

Geramos 100 tokens com `use_cache=False`. A cada novo token o modelo **recalcula Q, K, V de toda a sequencia** (recalculo redundante) e a atencao `eager` materializa a matriz O(n^2) na VRAM. Esperamos: **lento** e **alto pico de memoria**.

> ⚠️ **Se der erro de Out-Of-Memory aqui:** (1) primeiro **reinicie a sessao** (Ambiente de execucao -> Reiniciar sessao) e rode tudo desde o inicio — o OOM geralmente vem de memoria residual de uma execucao anterior; (2) se ainda assim estourar, reduza `TOKENS_ALVO` no Passo 2 para `5000` ou `4000`. O proprio estouro da atencao `eager` em contextos grandes e o "desastre O(n^2)" que o lab descreve.

In [ ]:
model.config.use_cache = False

limpar_memoria()
torch.cuda.synchronize()
inicio = time.time()

with torch.no_grad():
    saida_sem_cache = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=False,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
tempo_sem_cache = time.time() - inicio
pico_vram_sem_cache = torch.cuda.max_memory_allocated() / 1024**2

print(f"[METRICA 3] Tempo de geracao SEM cache: {tempo_sem_cache:.2f} s")
print(f"[METRICA 4] Pico de VRAM SEM cache (eager O(n^2)): {pico_vram_sem_cache:.2f} MB")

## Passo 4 — A Engenharia de Otimização (KV Cache + atenção eficiente)

Recarregamos o modelo com **SDPA** (`attn_implementation="sdpa"`) — a atencao eficiente em memoria do PyTorch, que, assim como o FlashAttention-2, **nao materializa a matriz O(n^2)** na VRAM (usa kernels otimizados em blocos). Ativamos tambem o **KV Cache** (`use_cache=True`), que elimina o recalculo do passado. Mesma geracao de 100 tokens.

> O FlashAttention-2 exigiria GPU Ampere+ (indisponivel na T4 gratuita). O SDPA pertence a mesma familia de otimizacao *hardware-aware* e e o substituto direto. Ver justificativa no README.

In [ ]:
# Libera o modelo eager antes de recarregar otimizado
del model
limpar_memoria()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    attn_implementation="sdpa",        # atencao eficiente em memoria (substituto do FA2 na T4)
)
model.config.use_cache = True

limpar_memoria()
torch.cuda.synchronize()
inicio = time.time()

with torch.no_grad():
    saida_otimizada = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
tempo_otimizado = time.time() - inicio
pico_vram_otimizado = torch.cuda.max_memory_allocated() / 1024**2

print(f"[METRICA 5] Tempo COM cache + SDPA: {tempo_otimizado:.2f} s")
print(f"[METRICA 6] Pico de VRAM otimizado: {pico_vram_otimizado:.2f} MB")

## Resumo do Benchmark + Gráficos.

In [ ]:
import matplotlib.pyplot as plt

speedup = tempo_sem_cache / tempo_otimizado
reducao_vram = (1 - pico_vram_otimizado / pico_vram_sem_cache) * 100

print("=" * 55)
print("RESUMO DO BENCHMARK ")
print("=" * 55)
print(f"GPU:                      {torch.cuda.get_device_name(0)}")
print(f"VRAM modelo 4-bit:        {vram_modelo:10.2f} MB")
print(f"Tokens de contexto:       {num_tokens:10d}")
print(f"Tempo SEM cache:          {tempo_sem_cache:10.2f} s")
print(f"Tempo COM cache + SDPA:   {tempo_otimizado:10.2f} s")
print(f"Pico VRAM SEM cache:      {pico_vram_sem_cache:10.2f} MB")
print(f"Pico VRAM otimizado:      {pico_vram_otimizado:10.2f} MB")
print(f"Speedup:                  {speedup:10.2f} x")
print(f"Reducao de VRAM:          {reducao_vram:10.2f} %")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
rotulos = ["Sem Cache\n(eager)", "Cache + SDPA"]
ax1.bar(rotulos, [tempo_sem_cache, tempo_otimizado], color=["#c0392b", "#27ae60"])
ax1.set_title("Tempo de geracao (100 tokens)"); ax1.set_ylabel("segundos")
ax2.bar(rotulos, [pico_vram_sem_cache, pico_vram_otimizado], color=["#c0392b", "#27ae60"])
ax2.set_title("Pico de VRAM"); ax2.set_ylabel("MB")
plt.tight_layout()
plt.savefig("benchmark.png", dpi=120)
plt.show()

## Amostra do texto gerado (apenas para inspeção)

In [ ]:
texto_gerado = tokenizer.decode(saida_otimizada[0][num_tokens:], skip_special_tokens=True)
print("Amostra dos 100 tokens gerados:\n")
print(texto_gerado)